In [1]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

In [22]:
class MyFastaParser:

    def __init__(self, filename):
        self.filename = filename

    def _get_uniprot(self, accession):
        url = f'https://rest.uniprot.org/uniprotkb/{accession}'
        return requests.get(url)

    def _get_ensembl(self, ensembl_id):
        url = f'https://rest.ensembl.org/lookup/id/{ensembl_id}'
        headers = {"Content-Type": "application/json"}
        return requests.get(url, headers=headers)

    def _uniprot_parse_response(self, response: requests.Response):
        data = response.json()
        organism = data.get('organism', {})

        if isinstance(organism, dict):
            organism_name = organism.get('scientificName', '')
        else:
            organism_name = organism

        return {
           'organism': organism_name,
           'geneInfo': data.get('genes', []),
           'sequenceInfo': data.get('sequence', {}),
           'type': 'protein'
        }

    def _ensembl_parse_response(self, response: requests.Response):
        data = response.json()
        to_return = ['object_type', 'species', 'assembly_name', 'biotype',
            'display_name', 'id', 'db_type', 'description',
            'source', 'canonical_transcript']

        return {key: data.get(key, '') for key in to_return}

    def _access_database(self, db_id, database, seq_description, seq_sequence) -> dict:
        if database == "uniprot":
            response = self._get_uniprot(db_id)
            if response.status_code == 200:
                return self._uniprot_parse_response(response)
        elif database == "ensembl":
            response = self._get_ensembl(db_id)
            if response.status_code == 200:
                return self._ensembl_parse_response(response)
        return {}

    def seqkit_stats(self) -> dict:
        try:
            result = subprocess.run(
                ['./seqkit.exe', 'stats', '-a', '-T', self.filename],
                capture_output=True, text=True, check=True
            )
            lines = result.stdout.strip().split('\n')
            if len(lines) < 2:
                return {"WARNING": "Unexpected seqkit output"}

            headers = lines[0].split('\t')
            values = lines[1].split('\t')

            stat_info = dict(zip(headers, values))
            if 'file' in stat_info:
                del stat_info['file']

            seq_type = stat_info.get('type', stat_info.get('Type', 'Unknown'))
            num_seqs = int(stat_info.get('num_seqs', 0))

            return {
                'fasta_seqkit_stat_info': stat_info,
                'fasta_type': seq_type,
                'fasta_num_seqs': num_seqs
            }
        except subprocess.CalledProcessError as e:
            return {"WARNING": e.stderr.strip() if e.stderr else str(e)}
        except FileNotFoundError:
             return {"WARNING": "seqkit command not found. Ensure it is installed."}


    def biopython_parser(self, seqkit_result) -> dict:
        if "WARNING" in seqkit_result:
            return {"WARNING": seqkit_result["WARNING"]}

        fasta_type = seqkit_result.get('fasta_type', '').upper()

        if 'PROTEIN' in fasta_type:
            db_name = "uniprot"
            pattern = re.compile(r'[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]{1,2}')
        else:
            db_name = "ensembl"
            pattern = re.compile(r'ENS[A-Z]*[0-9]{11}')

        output = {'DB_name': db_name}
        warnings = set()

        for record in SeqIO.parse(self.filename, "fasta"):
            desc = record.description
            seq = str(record.seq)

            match = pattern.search(desc)
            if match:
                db_id = match.group(0)
                output[f"file_info_{db_id}"] = {
                    "description": desc,
                    "sequence": seq
                }

                db_info = self._access_database(db_id, db_name, desc, seq)
                if db_info:
                    output[f"database_info_{db_id}"] = db_info
            else:
                warnings.add('No ID match found.')

        if warnings:
            output['WARNING'] = warnings

        return output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [7]:
parser = MyFastaParser('test_file.fasta')

In [8]:
stats = parser.seqkit_stats()

In [9]:
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228.0',
  'max_len': '427',
  'Q1': '29',
  'Q2': '228',
  'Q3': '427',
  'sum_gap': '0',
  'N50': '427',
  'N50_num': '1',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '0.00',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [10]:
biopython = parser.biopython_parser(stats)

In [11]:
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS

In [12]:
parser = MyFastaParser('uniprot_download.fasta')

In [13]:
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '7',
  'sum_len': '3861',
  'min_len': '180',
  'avg_len': '551.6',
  'max_len': '1382',
  'Q1': '429',
  'Q2': '441',
  'Q3': '500',
  'sum_gap': '0',
  'N50': '468',
  'N50_num': '3',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '0.00',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 7}

In [14]:
biopython = parser.biopython_parser(stats)

In [15]:
parser.show_output(biopython)

DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
		[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
	sequenceInfo
		value
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLL

In [18]:
parser = MyFastaParser('ensembl_download_1.fasta')

In [19]:
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'DNA',
  'num_seqs': '6',
  'sum_len': '86',
  'min_len': '9',
  'avg_len': '14.3',
  'max_len': '23',
  'Q1': '10',
  'Q2': '14',
  'Q3': '17',
  'sum_gap': '0',
  'N50': '16',
  'N50_num': '3',
  'Q20(%)': '0',
  'Q30(%)': '0',
  'AvgQual': '0.00',
  'GC(%)': '45.35',
  'sum_n': '0'},
 'fasta_type': 'DNA',
 'fasta_num_seqs': 6}

In [20]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221
	object_type
		Transcript
	species
		mus_musculus
	assembly_name
		GRCm39
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST00000196221
	db_type
		core
	description
		
	source
		havana
	canonical_transcript
		
file_info_ENSMUST00000177564
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
	sequence
		ATCGGAGGGATACGAG
database_info_ENSMUST00000177564
	object_type
		Transcript
	species
		mus_musculus
	assembly_name
		GRC

In [21]:
parser = MyFastaParser('ensembl_download_2.fasta')
stats = parser.seqkit_stats()
stats

{'WARNING': 'Unexpected seqkit output'}